# DiscoveryStack SEO/GEO 500：evidence-aware 多任務訓練

> 本模板只接受 owner-approved `manifest-v4-500`。所有 primaryStage 與 stage evidence 必須經人工覆核；這仍是 development candidate，不是 production model。

本版本保留共享文字 encoder，新增只供 `journeyStage` 使用的 `stageCueVector` branch，並以 validation journeyStage macro-F1、per-class recall、zero-prediction gate 與 legacy test regression 選模。

In [ ]:
!pip -q install transformers scikit-learn sentencepiece safetensors
!nvidia-smi

EXPECTED_ROW_COUNT = 500
EXPECTED_MANIFEST_HASH = '<fill-from-approved-manifest>'
EXPECTED_DATASET_DIGEST = '<fill-from-approved-manifest>'
EXPECTED_SPLITS = {'train': 350, 'validation': 75, 'test_v2': 75}
PRIMARY_STAGES = ['discovery', 'understanding', 'response', 'progression', 'conversion']
FEATURE_CONTRACT_VERSION = 'features-v1'
MODEL_VERSION = 'seo-geo-multitask-colab-v4'
SEEDS = [20260820, 20260821, 20260822]

In [ ]:
# Load only the owner-private v4 snapshot; do not run the upload fallback in the same runtime.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from io import FileIO
from pathlib import Path

DRIVE_SNAPSHOT_NAME = 'discoverystack-manifest-v4-500-private.jsonl'
DATA_PATH = Path('/content/discoverystack-manifest-v4-500.jsonl')
auth.authenticate_user()
drive_service = build('drive', 'v3')
matches = drive_service.files().list(q=f"name = '{DRIVE_SNAPSHOT_NAME}' and trashed = false", spaces='drive', fields='files(id,name,size,trashed)').execute().get('files', [])
assert len(matches) == 1, f'FAIL-CLOSED: expected one private v4 snapshot, found {len(matches)}'
request = drive_service.files().get_media(fileId=matches[0]['id'])
with FileIO(DATA_PATH, 'wb') as target:
    downloader = MediaIoBaseDownload(target, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
assert DATA_PATH.stat().st_size > 100_000

In [ ]:
# Fail-closed schema, governance, group split and stage balance checks.
import hashlib, json
from collections import Counter
raw_lines = [line.rstrip('\n') for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
rows = [json.loads(line) for line in raw_lines]
assert len(rows) == EXPECTED_ROW_COUNT
assert hashlib.sha256('\n'.join(raw_lines).encode()).hexdigest() == EXPECTED_DATASET_DIGEST
assert len({int(row['id']) for row in rows}) == EXPECTED_ROW_COUNT
assert Counter(row['split'] for row in rows) == Counter(EXPECTED_SPLITS)
assert min(Counter(row['targets']['journeyStage'] for row in rows).values()) >= 80
for row in rows:
    assert row['reviewState'] in {'reviewed', 'needs_adjudication'} and row['labelMethod'] in {'human', 'human_amended', 'assistant_rule_candidate', 'owner_approved_legacy'}
    assert row['taxonomyVersion'] == 'journey-v3' and row['featureContractVersion'] == FEATURE_CONTRACT_VERSION
    assert row['governance']['rightsStatus'] == 'approved'
    assert row['governance']['robotsChecked'] is True
    assert row['governance']['piiStatus'] in {'none_detected', 'masked'}
    assert len(row['stageEvidence']) >= 1
print({'rows': len(rows), 'splits': dict(Counter(row['split'] for row in rows)), 'stages': dict(Counter(row['targets']['journeyStage'] for row in rows))})

In [ ]:
# Reconstruct inference-safe stageCueVector from text and approved structured extracts.
import re
CUE_PATTERNS = {
    'problem_statement_count': r'\b(problem|challenge|why|issue)\b',
    'question_heading_count': r'\?',
    'definition_cue_count': r'\b(what is|definition|means|introduction)\b',
    'how_it_works_cue_count': r'\b(how it works|steps|guide|tutorial)\b',
    'comparison_cue_count': r'\b(compare|comparison|versus|pros|cons|alternative)\b',
    'requirements_cue_count': r'\b(requirement|prerequisite|eligibility)\b',
    'troubleshooting_cue_count': r'\b(troubleshoot|troubleshooting|fix|resolve)\b',
    'error_debug_cue_count': r'\b(error|debug|incorrect|failure)\b',
    'remediation_cue_count': r'\b(remediation|repair|改善|修復)\b',
    'cta_count': r'\b(contact|book|buy|sign up|get started|request)\b',
    'contact_purchase_cue_count': r'\b(contact|purchase|order|checkout|shipping|returns)\b'
}
def make_stage_features(row):
    text = row['trainingText'].lower()
    vector = {name: float(len(re.findall(pattern, text))) for name, pattern in CUE_PATTERNS.items()}
    vector['form_presence'] = float(bool(re.search(r'\b(form|email|phone)\b', text)))
    return vector
stage_features = [make_stage_features(row) for row in rows]
print({'featureContractVersion': FEATURE_CONTRACT_VERSION, 'featureCount': len(stage_features[0])})

In [ ]:
# Evidence-aware architecture: stage branch only; other heads stay text-only.
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiTaskModelV4(nn.Module):
    def __init__(self, model_id, task_label_maps, stage_feature_dim):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden = self.encoder.config.hidden_size
        self.stage_feature_mlp = nn.Sequential(nn.LayerNorm(stage_feature_dim), nn.Linear(stage_feature_dim, 64), nn.GELU(), nn.Dropout(0.1))
        self.heads = nn.ModuleDict({task: nn.Linear(hidden + (64 if task == 'journeyStage' else 0), len(task_label_maps[task])) for task in task_label_maps})
    def forward(self, input_ids, attention_mask, stage_features):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        stage_input = torch.cat([pooled, self.stage_feature_mlp(stage_features)], dim=-1)
        return {task: self.heads[task](stage_input if task == 'journeyStage' else pooled) for task in self.heads}

In [ ]:
# Stage-weighted loss; fit class weights on train only.
import numpy as np
stage_loss_weight = 2.0
stage_class_weight = torch.tensor([...], dtype=torch.float32, device=DEVICE)  # inverse-sqrt train frequencies
def compute_loss_v4(logits, labels, losses):
    total = stage_loss_weight * nn.functional.cross_entropy(logits['journeyStage'], labels['journeyStage'], weight=stage_class_weight)
    for task in MULTI_LABEL_TASKS + ['actionPriority']:
        total = total + losses[task](logits[task], labels[task])
    return total / (stage_loss_weight + len(MULTI_LABEL_TASKS) + 1)

## Evaluation and packaging gate

選模只看 validation `journeyStage.macroF1` 與 per-class recall；test 只在所有 seed 完成後執行一次。任一 `discovery`／`response` 預測支持數為零、legacy test regression 退化或 artifact reload smoke test 失敗，均標記 `candidate_not_ready`。最後使用 repo 的 `ml/packaging/package_artifact.py` 生成 allow-list ZIP、artifact manifest 與 `checksums/SHA256SUMS`；原始 JSONL 與 HTML 保留在 owner-only Drive，不進 ZIP。